# 🎙️ Hindi TTS — Self-Contained GPU Training Pipeline
### Works 100% standalone — no broken scripts, no missing deps, no HF token needed

| Step | What it does | Time (T4 GPU) |
|------|--------------|---------------|
| 1 | Check GPU | 5 s |
| 2 | Download audio from Google Drive | 1–2 min |
| 3 | Install all dependencies (fixed) | 3–5 min |
| 4 | Whisper transcription + auto-chunking | 20–40 min |
| 5 | Build GPT-SoVITS filelist (inline, no broken scripts) | 10 s |
| 6 | Clone & launch GPT-SoVITS WebUI (all deps pre-fixed) | 5–10 min |
| Bonus | Test Edge-TTS instant Hindi speech | 5 s |

> ⚠️ Set **Runtime → Change runtime type → T4 GPU** before running.

## ⚡ Step 1: Check GPU

In [1]:
import torch
print('='*55)
print('GPU Available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Name      :', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM          : {vram:.1f} GB')
else:
    print('⚠️  NO GPU — go to Runtime → Change runtime type → T4 GPU')
print('='*55)
!nvidia-smi

GPU Available : True
GPU Name      : Tesla T4
VRAM          : 15.6 GB
Tue Nov 10 13:00:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |  

## 📁 Step 2: Get Audio Files from Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Where audio ends up (regardless of source) ───────────────────────────────
AUDIO_DIR = '/content/hindi_tts/audio'
os.makedirs(AUDIO_DIR, exist_ok=True)

DRIVE_FOLDER = '/content/drive/MyDrive/hindi_tts_data'

if os.path.exists(DRIVE_FOLDER):
    import shutil
    wav_files = [f for f in os.listdir(DRIVE_FOLDER) if f.lower().endswith('.wav')]
    for f in wav_files:
        dest = os.path.join(AUDIO_DIR, f)
        if not os.path.exists(dest):
            shutil.copy2(os.path.join(DRIVE_FOLDER, f), dest)
    print(f'[+] Copied {len(wav_files)} WAV files from Drive → {AUDIO_DIR}')
else:
    print('[-] Drive folder not found — using gdown fallback...')
    !pip install -q gdown
    import gdown
    gdown.download_folder(
        'https://drive.google.com/drive/folders/17OKMSInQ7-8KFNUn0voAYPhFzViLAFra?usp=sharing',
        output=AUDIO_DIR, quiet=False
    )

wav_files = [f for f in os.listdir(AUDIO_DIR) if f.lower().endswith('.wav')]
print(f'\n[+] Total WAV files ready : {len(wav_files)}')
for f in wav_files[:4]:
    mb = os.path.getsize(os.path.join(AUDIO_DIR, f)) / 1e6
    print(f'    {mb:5.1f} MB  {f[:60]}')
if len(wav_files) > 4:
    print(f'    ... and {len(wav_files)-4} more')

Mounted at /content/drive
[-] Drive folder not found — using gdown fallback...


Retrieving folder contents


Processing file 1rRZi0cAD-bjK_Na2HK8N-x7M74zeuByA Baagh Aaya Bhago! 😨 ｜ Pintu Bhaiya vs Leopard ｜ Hindi Cartoon Story.wav
Processing file 14UMKvUh71XZlqL8G3mrhSKm2Jia7Likz Gaon Main Gas Ka Ghotala ｜ गाँव में गैस का घोटाला  ｜ New Hindi Kahani ｜ Moral Story.wav
Processing file 1MuLKyySW3BynNE9bbcujHzmDAwx6MkJX Holi Ka Rang ｜ Pintu Aur Mukhiya Ki Funny Holi ｜ Hindi Comedy Cartoon Story.wav
Processing file 1rQXJ2bwGj9SJ1KvGdb12gnAH1HZBovku Horror Story 😨 - कर्णेश्वर राक्षस का खजाना ｜ Horror Story ｜ Kahani Junction ｜ #horrorstories.wav
Processing file 13CjRIReqbpjZFTV3xVNp3ZNazEkkDiFE Horror Story 😨 - घुंघुरू की आवाज ｜ True Horror Stories ｜ Kahani Junction ｜ #horrorstories.wav
Processing file 1BnNNCiGfxX77ZNVAcNo8DtFoOe9MuihH Horror Story 😨 - भूतिया रेलवे स्टेशन ｜ Haunted Railway Station ｜ Kahani Junction ｜ #hauntedstation.wav
Processing file 1PWJr12dUQ5fev39lKVFRZgD4pYqELwC6 Horror Story 😨 - भूतिया लड़का ｜ Story of a Ghost boy ｜ Kahani Junction ｜ #horrorstories.wav
Processing file 130jLLdI

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1rRZi0cAD-bjK_Na2HK8N-x7M74zeuByA
To: /content/hindi_tts/audio/Baagh Aaya Bhago! 😨 ｜ Pintu Bhaiya vs Leopard ｜ Hindi Cartoon Story.wav
100%|██████████| 19.4M/19.4M [00:00<00:00, 140MB/s] 
Downloading...
From: https://drive.google.com/uc?id=14UMKvUh71XZlqL8G3mrhSKm2Jia7Likz
To: /content/hindi_tts/audio/Gaon Main Gas Ka Ghotala ｜ गाँव में गैस का घोटाला  ｜ New Hindi Kahani ｜ Moral Story.wav
100%|██████████| 22.2M/22.2M [00:00<00:00, 175MB/s]
Downloading...
From: https://drive.google.com/uc?id=1MuLKyySW3BynNE9bbcujHzmDAwx6MkJX
To: /content/hindi_tts/audio/Holi Ka Rang ｜ Pintu Aur Mukhiya Ki Funny Holi ｜ Hindi Comedy Cartoon Story.wav
100%|██████████| 24.7M/24.7M [00:00<00:00, 110MB/s]
Downloading...
From: https://drive.google.com/uc?id=1rQXJ2bwGj9SJ1KvGdb12gnAH1HZBovku
To: /content/hindi_tts/audio/Horror Story 😨 - कर्णेश्वर राक्षस का ख


[+] Total WAV files ready : 26
     24.7 MB  Holi Ka Rang ｜ Pintu Aur Mukhiya Ki Funny Holi ｜ Hindi Comed
     19.4 MB  Baagh Aaya Bhago! 😨 ｜ Pintu Bhaiya vs Leopard ｜ Hindi Cartoo
     20.1 MB  एक चिपकली और पूरे गाँव की जीत ｜ Comedy Hindi Story ｜ New Hin
      7.0 MB  Horror Story 😨 - भूतिया लड़का ｜ Story of a Ghost boy ｜ Kahan
    ... and 22 more



Download completed


## 📦 Step 3: Install ALL Dependencies (Fixed)

Fixes applied vs the old notebook:
- Added `cmake` + system build tools so `opencc` compiles
- Added `mecab` so `python_mecab_ko` compiles  
- Added `ffmpeg-python` (the `import ffmpeg` GPT-SoVITS needs)
- Added `edge-tts`, `openai`, `pandas` which were missing

In [3]:
print('[1/4] Installing system build deps (cmake, mecab, ffmpeg)...')
!apt-get install -qq -y cmake mecab libmecab-dev mecab-ipadic-utf8 ffmpeg > /dev/null 2>&1
print('[+] System deps OK')

print('[2/4] Installing Python audio + TTS packages...')
!pip install -q \
    edge-tts \
    openai \
    openai-whisper \
    soundfile \
    pydub \
    ffmpeg-python \
    gdown \
    pandas \
    python-dotenv
print('[+] Audio + TTS packages OK')

print('[3/4] Installing demucs (vocal separation)...')
!pip install -q demucs
print('[+] Demucs OK')

print('[4/4] Verifying key imports...')
import whisper, soundfile, ffmpeg, edge_tts
print('[+] All imports OK!')
print('\n✅ Dependencies installed successfully!')

[1/4] Installing system build deps (cmake, mecab, ffmpeg)...
[+] System deps OK
[2/4] Installing Python audio + TTS packages...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
[+] Audio + TTS packages OK
[3/4] Installing demucs (vocal separation)...
[+] Demucs OK
[4/4] Verifying key imports...
[+] All imports OK!

✅ Dependencies installed successfully!


## 🎙️ Step 4: Whisper Transcription + Auto-Chunking

**Replaces the broken 3-script diarization pipeline** with a single step:
- Uses Whisper `medium` on each raw WAV
- Automatically splits into short segments using Whisper's own timestamps
- Saves each segment as a separate WAV file  
- Writes `metadata.csv` with correct local paths

> ⏱ ~20–40 min for all 26 stories on T4 GPU  
> To test faster, set `MAX_FILES = 3` below.

In [4]:
import os, csv, torch
import numpy as np
import soundfile as sf
import whisper

# ── Config ────────────────────────────────────────────────────────────────────
AUDIO_DIR    = '/content/hindi_tts/audio'
SEGMENTS_DIR = '/content/hindi_tts/segments'
METADATA_CSV = '/content/hindi_tts/metadata.csv'
WHISPER_MODEL = 'medium'  # 'small' for faster test, 'large-v3' for best quality
MIN_DURATION  = 2.0       # seconds — skip very short clips
MAX_DURATION  = 15.0      # seconds — split longer clips
MAX_FILES     = None      # Set to 3 for a quick test; None = all files

os.makedirs(SEGMENTS_DIR, exist_ok=True)

# ── Load Whisper ──────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[*] Loading Whisper {WHISPER_MODEL} on {device.upper()}...')
print('    (First run downloads ~1.5 GB model)')
model = whisper.load_model(WHISPER_MODEL, device=device)
print(f'[+] Whisper ready!\n')

# ── Helper: load + resample to 16kHz mono ─────────────────────────────────────
def load_audio_16k(path):
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)   # stereo → mono
    if sr != 16000:
        import resampy
        audio = resampy.resample(audio, sr, 16000)
    return audio.astype(np.float32), 16000

# ── Helper: save a segment slice ──────────────────────────────────────────────
def save_segment(audio, sr, start, end, out_path):
    s, e = int(start * sr), int(end * sr)
    chunk = audio[s:e]
    if len(chunk) == 0:
        return False
    sf.write(out_path, chunk, sr)
    return True

# ── Main loop ─────────────────────────────────────────────────────────────────
wav_files = sorted([f for f in os.listdir(AUDIO_DIR) if f.lower().endswith('.wav')])
if MAX_FILES:
    wav_files = wav_files[:MAX_FILES]

print(f'[*] Processing {len(wav_files)} WAV files...\n')

records = []
total_files = len(wav_files)

for file_idx, fname in enumerate(wav_files, 1):
    story_name = os.path.splitext(fname)[0][:60]  # truncate for folder name
    wav_path   = os.path.join(AUDIO_DIR, fname)
    story_dir  = os.path.join(SEGMENTS_DIR, story_name)
    os.makedirs(story_dir, exist_ok=True)

    print(f'[{file_idx}/{total_files}] Transcribing: {fname[:70]}')

    try:
        # Transcribe with word-level timestamps
        result = model.transcribe(
            wav_path,
            language='hi',
            task='transcribe',
            word_timestamps=False,
            fp16=(device == 'cuda'),
            verbose=False,
        )
    except Exception as e:
        print(f'  [-] Whisper failed: {e}')
        continue

    # Load audio for slicing
    try:
        audio, sr = load_audio_16k(wav_path)
    except Exception as e:
        # Fallback: use whisper's own loader
        try:
            audio = whisper.load_audio(wav_path)
            sr = 16000
        except Exception as e2:
            print(f'  [-] Audio load failed: {e2}')
            continue

    file_segments = 0
    for seg in result.get('segments', []):
        text     = seg['text'].strip()
        start    = seg['start']
        end      = seg['end']
        duration = end - start

        if not text or duration < MIN_DURATION:
            continue

        # If segment is too long, split at midpoint
        if duration > MAX_DURATION:
            mid = (start + end) / 2
            sub_segments = [(start, mid), (mid, end)]
        else:
            sub_segments = [(start, end)]

        for sub_start, sub_end in sub_segments:
            seg_name = f'seg_{file_segments:04d}_{int(sub_start)}s.wav'
            seg_path = os.path.join(story_dir, seg_name)

            if save_segment(audio, sr, sub_start, sub_end, seg_path):
                records.append({
                    'audio_path': seg_path,
                    'speaker_id': 'SPEAKER_00',
                    'story':      story_name,
                    'language':   'hi',
                    'text':       text,
                    'duration':   round(sub_end - sub_start, 2),
                })
                file_segments += 1

    print(f'  [+] {file_segments} segments → {story_dir}')

# ── Write metadata.csv ────────────────────────────────────────────────────────
with open(METADATA_CSV, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['audio_path','speaker_id','story','language','text','duration'])
    writer.writeheader()
    writer.writerows(records)

print(f'\n✅ Transcription complete!')
print(f'   Total segments : {len(records)}')
print(f'   Metadata CSV   : {METADATA_CSV}')

[*] Loading Whisper medium on CUDA...
    (First run downloads ~1.5 GB model)


100%|█████████████████████████████████████| 1.42G/1.42G [00:16<00:00, 93.0MiB/s]


[+] Whisper ready!

[*] Processing 26 WAV files...

[1/26] Transcribing: Baagh Aaya Bhago! 😨 ｜ Pintu Bhaiya vs Leopard ｜ Hindi Cartoon Story.wa


 95%|█████████▌| 57517/60517 [05:54<00:18, 162.33frames/s]


  [+] 98 segments → /content/hindi_tts/segments/Baagh Aaya Bhago! 😨 ｜ Pintu Bhaiya vs Leopard ｜ Hindi Cartoo
[2/26] Transcribing: Gaon Main Gas Ka Ghotala ｜ गाँव में गैस का घोटाला  ｜ New Hindi Kahani 


 88%|████████▊ | 60966/69324 [09:44<01:20, 104.31frames/s]


... (45 outputs left out)

100%|██████████| 28042/28042 [03:22<00:00, 138.81frames/s]


  [+] 62 segments → /content/hindi_tts/segments/भूतिया लाइब्रेरी - True Horror Stories ｜ Bhoot Ki Kahani ｜ D
[26/26] Transcribing: हाईजैक बस  😱 ｜ Hijack Bus Story ｜ Pintu Hero ｜ Pintu Ka Master Plan ｜ 


 95%|█████████▍| 58610/61716 [06:12<00:19, 157.34frames/s]


  [+] 122 segments → /content/hindi_tts/segments/हाईजैक बस  😱 ｜ Hijack Bus Story ｜ Pintu Hero ｜ Pintu Ka Mast

✅ Transcription complete!
   Total segments : 3385
   Metadata CSV   : /content/hindi_tts/metadata.csv


## 📊 Step 4B: Preview Dataset

In [5]:
import pandas as pd

df = pd.read_csv('/content/hindi_tts/metadata.csv')
print(f'Total segments : {len(df)}')
print(f'Stories        : {df["story"].nunique()}')
print(f'Total duration : {df["duration"].sum()/60:.1f} min')
print(f'Avg duration   : {df["duration"].mean():.2f} s')
print()
print('Duration buckets:')
print(f'  < 2 s  : {(df.duration < 2).sum()}')
print(f'  2–8 s  : {((df.duration >= 2) & (df.duration < 8)).sum()}')
print(f'  8–15 s : {((df.duration >= 8) & (df.duration <= 15)).sum()}')
print()
print(df[['story','text','duration']].head(8).to_string(index=False))

Total segments : 3385
Stories        : 26
Total duration : 252.1 min
Avg duration   : 4.47 s

Duration buckets:
  < 2 s  : 0
  2–8 s  : 2971
  8–15 s : 414

                                                       story                                                       text  duration
Baagh Aaya Bhago! 😨 ｜ Pintu Bhaiya vs Leopard ｜ Hindi Cartoo                                                     तुम्मु       2.0
Baagh Aaya Bhago! 😨 ｜ Pintu Bhaiya vs Leopard ｜ Hindi Cartoo                                                     तुम्मु       2.0
Baagh Aaya Bhago! 😨 ｜ Pintu Bhaiya vs Leopard ｜ Hindi Cartoo       चुट्की, बुट्टन, तुम स्कूल जाओ, मैं बाद में मिलते हूं       3.0
Baagh Aaya Bhago! 😨 ｜ Pintu Bhaiya vs Leopard ｜ Hindi Cartoo         ये पिंटू भिया के वजह से आज भी आमें एकेले जाना होगा       3.0
Baagh Aaya Bhago! 😨 ｜ Pintu Bhaiya vs Leopard ｜ Hindi Cartoo      इनके पिताजी सही कहते है, पूरा दिन आवारा बनकर गूनते है       4.0
Baagh Aaya Bhago! 😨 ｜ Pintu Bhaiya vs Leopard ｜ Hindi Cartoo अर

## 🤖 Step 5: Build GPT-SoVITS Training Filelist (Inline — No Broken Scripts)

Builds `sovits_filelist.txt` and `xtts_metadata.csv` directly from `metadata.csv`.
No dependency on `scripts/08_format_sovits.py`.

In [6]:
import pandas as pd, os, json, torch

METADATA_CSV  = '/content/hindi_tts/metadata.csv'
OUTPUT_DIR    = '/content/hindi_tts/training_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(METADATA_CSV)

# ── Filter: keep only 2–15 s segments with non-empty text ────────────────────
df = df[
    df['text'].notna() &
    (df['text'].str.strip() != '') &
    (df['audio_path'].apply(os.path.exists)) &
    (df['duration'] >= 2.0) &
    (df['duration'] <= 15.0)
].reset_index(drop=True)

print(f'[+] Filtered dataset: {len(df)} usable segments')

# ── GPT-SoVITS filelist: audio_path|speaker|lang|text ────────────────────────
SOVITS_FILELIST = os.path.join(OUTPUT_DIR, 'sovits_filelist.txt')
lines = []
for _, row in df.iterrows():
    speaker = row.get('speaker_id', 'SPEAKER_00')
    lines.append(f"{row['audio_path']}|{speaker}|HI|{row['text'].strip()}")
with open(SOVITS_FILELIST, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f'[+] GPT-SoVITS filelist → {SOVITS_FILELIST}  ({len(lines)} lines)')

# ── XTTS-v2 metadata CSV ──────────────────────────────────────────────────────
XTTS_META = os.path.join(OUTPUT_DIR, 'xtts_metadata.csv')
xtts_lines = ['audio_file|text|speaker_name']
for _, row in df.iterrows():
    speaker = row.get('speaker_id', 'SPEAKER_00')
    xtts_lines.append(f"{row['audio_path']}|{row['text'].strip()}|{speaker}")
with open(XTTS_META, 'w', encoding='utf-8') as f:
    f.write('\n'.join(xtts_lines))

# ── XTTS config (auto-detects cuda/cpu) ───────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
xtts_config = {
    'run_name': 'hindi_tts_xtts',
    'model': 'xtts',
    'audio': {'sample_rate': 22050, 'max_wav_value': 32768.0},
    'datasets': [{
        'name': 'hindi_tts',
        'path': OUTPUT_DIR,
        'meta_file': 'xtts_metadata.csv',
        'language': 'hi'
    }],
    'training': {
        'batch_size': 4,
        'epochs': 100,
        'learning_rate': 0.0001,
        'mixed_precision': device == 'cuda',
        'device': device,
    }
}
XTTS_CONFIG = os.path.join(OUTPUT_DIR, 'xtts_config.json')
with open(XTTS_CONFIG, 'w') as f:
    json.dump(xtts_config, f, indent=2, ensure_ascii=False)

print(f'[+] XTTS metadata    → {XTTS_META}')
print(f'[+] XTTS config      → {XTTS_CONFIG}  (device={device})')
print()
print('✅ Training data ready!')
print(f'   Segments : {len(df)}')
print(f'   Duration : {df["duration"].sum()/60:.1f} min total')

[+] Filtered dataset: 3385 usable segments
[+] GPT-SoVITS filelist → /content/hindi_tts/training_data/sovits_filelist.txt  (3385 lines)
[+] XTTS metadata    → /content/hindi_tts/training_data/xtts_metadata.csv
[+] XTTS config      → /content/hindi_tts/training_data/xtts_config.json  (device=cuda)

✅ Training data ready!
   Segments : 3385
   Duration : 252.1 min total


In [15]:
webui_path = "/content/GPT-SoVITS/webui.py"
with open(webui_path) as f:
    content = f.read()

content = content.replace(
    "        share=is_share,\n        server_port=webui_port_main,",
    "        server_port=webui_port_main,"
)

with open(webui_path, "w") as f:
    f.write(content)

import subprocess
r = subprocess.run(["python", "-m", "py_compile", webui_path], capture_output=True, text=True)
print("[+] FIXED!" if r.returncode == 0 else "[-] " + r.stderr[:300])

[+] FIXED!


## 🚀 Step 6: Install & Launch GPT-SoVITS Training WebUI

**Fixes applied vs old notebook:**
1. `cmake` pre-installed → `opencc` builds successfully
2. `mecab` pre-installed → `python_mecab_ko` builds successfully  
3. `ffmpeg-python` installed → `import ffmpeg` works in `webui.py`
4. `filelist.txt` already copied into GPT-SoVITS dir

> 👉 After the cell runs, click the **`*.gradio.live`** link.  
> In the WebUI: set **Training filelist** to `/content/GPT-SoVITS/hindi_filelist.txt`

In [ ]:
import os, re, subprocess

# --- 6a: Clone GPT-SoVITS ---
print("[1/5] Cloning GPT-SoVITS...")
!git clone https://github.com/RVC-Boss/GPT-SoVITS.git /content/GPT-SoVITS 2>/dev/null || echo "[*] Already cloned"
os.chdir("/content/GPT-SoVITS")
print("[+] Cloned OK")

# --- 6b: Install requirements ---
print("[2/5] Installing GPT-SoVITS requirements...")
!grep -Ev "opencc|python.mecab" requirements.txt | pip install -q -r /dev/stdin 2>&1 | tail -5
!pip install -q opencc-python-reimplemented ffmpeg-python 2>&1 | tail -3
print("[+] Requirements installed")

# --- 6c: FIX webui.py duplicate share= keyword ---
# Root cause (now confirmed from debug output):
#   L1977: app.queue().launch(share=True,   # share= is ON THE SAME LINE as .launch(
#   L1980:     share=is_share,              # <- duplicate
# Previous parser did "continue" on the .launch( line without registering share=True.
# Fix: when we detect .launch(, also scan THAT line for share= before continuing.
print("[3/5] Patching webui.py (fixed parser)...")
webui_path = "/content/GPT-SoVITS/webui.py"
with open(webui_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Pass 1: set is_share = True wherever it is defined
patched = []
for l in lines:
    l = re.sub(r"is_share\s*=\s*False", "is_share = True", l)
    l = re.sub(r"is_share\s*=\s*os\.environ[^\n]*", "is_share = True\n", l)
    l = re.sub(r"is_share\s*=\s*args\.[^\n]*",    "is_share = True\n", l)
    patched.append(l)

# Pass 2: deduplicate share= args inside .launch() blocks
# KEY FIX: when .launch( is detected, check if share= is ALSO on that same line
result = []
in_launch  = False
depth      = 0
share_seen = False
removed    = []

for idx, l in enumerate(patched, 1):
    # Detect .launch( on this line
    if ".launch(" in l and not in_launch:
        in_launch  = True
        depth      = l.count("(") - l.count(")")
        share_seen = False
        # CRITICAL FIX: check if share= is ALSO on the .launch( line itself
        if re.search(r"share\s*=", l):
            share_seen = True
        result.append(l)
        if depth <= 0: in_launch = False
        continue

    if in_launch:
        depth += l.count("(") - l.count(")")
        s = l.strip().rstrip(",")
        is_share_arg = bool(re.match(r"share\s*=", s))

        if is_share_arg:
            if share_seen:
                # Duplicate share= — drop this line
                removed.append((idx, l.rstrip()))
                if depth <= 0: in_launch = False
                continue
            else:
                share_seen = True

        if depth <= 0: in_launch = False

    result.append(l)

print(f"  Removed {len(removed)} duplicate share= line(s):")
for ln, text in removed:
    print(f"    L{ln}: {text}")

with open(webui_path, "w", encoding="utf-8") as f:
    f.writelines(result)

# Mandatory syntax check
check = subprocess.run(["python", "-m", "py_compile", webui_path], capture_output=True, text=True)
if check.returncode == 0:
    print("[+] webui.py syntax OK!")
else:
    print("[-] STILL broken:")
    print(check.stderr[:400])
    raise SystemExit("Stopping.")

# --- 6d: Copy filelist ---
print("[4/5] Copying Hindi filelist...")
src_fl = "/content/hindi_tts/training_data/sovits_filelist.txt"
dst_fl = "/content/GPT-SoVITS/hindi_filelist.txt"
if os.path.exists(src_fl):
    import shutil; shutil.copy(src_fl, dst_fl)
    n = len(open(dst_fl).readlines())
    print(f"[+] Filelist ready: {n} segments")
else:
    print("[-] Filelist missing -- run Step 5 first!")

# --- 6e: Launch ---
print("[5/5] Launching GPT-SoVITS WebUI (Gradio public URL in ~20s)...")
print()
print("  Training filelist -> /content/GPT-SoVITS/hindi_filelist.txt")
print()
!python webui.py


[1/5] Cloning GPT-SoVITS...
[*] Already cloned
[+] Cloned OK
[2/5] Installing GPT-SoVITS requirements...
[+] Requirements installed
[3/5] Patching webui.py with line-by-line parser...
[+] webui.py syntax OK -- SyntaxError fixed!
[4/5] Copying Hindi filelist...
[+] Filelist ready: 3385 segments
[5/5] Launching GPT-SoVITS WebUI (Gradio public URL in ~20s)...

  Training filelist -> /content/GPT-SoVITS/hindi_filelist.txt

Running on local URL:  http://0.0.0.0:9874
ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await se

---
## 💾 Save Outputs to Google Drive (Run After Training)

In [8]:
import shutil, os
from google.colab import drive

# Make sure Drive is mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

SAVE_TO = '/content/drive/MyDrive/hindi_tts_outputs'
os.makedirs(SAVE_TO, exist_ok=True)

# Save metadata + filelist
shutil.copy('/content/hindi_tts/metadata.csv',
            os.path.join(SAVE_TO, 'metadata.csv'))
shutil.copy('/content/hindi_tts/training_data/sovits_filelist.txt',
            os.path.join(SAVE_TO, 'sovits_filelist.txt'))

# Save GPT-SoVITS model checkpoints if they exist
CKPT_DIRS = [
    '/content/GPT-SoVITS/logs',
    '/content/GPT-SoVITS/SoVITS_weights',
    '/content/GPT-SoVITS/GPT_weights',
]
for d in CKPT_DIRS:
    if os.path.exists(d):
        dest = os.path.join(SAVE_TO, os.path.basename(d))
        shutil.copytree(d, dest, dirs_exist_ok=True)
        print(f'[+] Saved {d} → {dest}')

print(f'\n✅ All outputs saved to Google Drive: {SAVE_TO}')

[+] Saved /content/GPT-SoVITS/SoVITS_weights → /content/drive/MyDrive/hindi_tts_outputs/SoVITS_weights
[+] Saved /content/GPT-SoVITS/GPT_weights → /content/drive/MyDrive/hindi_tts_outputs/GPT_weights

✅ All outputs saved to Google Drive: /content/drive/MyDrive/hindi_tts_outputs


---
## 🎧 Bonus: Instant Hindi TTS Test (Edge-TTS — No Training Needed)

In [9]:
import asyncio, os
import edge_tts
from IPython.display import Audio, display

TEXT   = 'नमस्ते! यह हिंदी टेक्स्ट-टू-स्पीच का परीक्षण है। आज का दिन बहुत अच्छा है।'
VOICE  = 'hi-IN-MadhurNeural'   # male narrator
OUTPUT = '/content/hindi_tts/test_tts.mp3'

async def gen():
    comm = edge_tts.Communicate(text=TEXT, voice=VOICE, rate='+0%', pitch='+0Hz')
    await comm.save(OUTPUT)

await gen()
print(f'[+] Audio saved: {OUTPUT}')
display(Audio(OUTPUT, autoplay=True))

[+] Audio saved: /content/hindi_tts/test_tts.mp3
